In [3]:
import pandas as pd
df=pd.read_csv('real_india_fire_aqi_dataset_uttarakhand.csv')
df.head()

,latitude,longitude,date,air_temp_c,rel_humidity_pct,wind_speed_kmh,wind_gust_kmh,surface_pressure_hpa,precipitation_mm,soil_moisture_vol,...,elevation_m,sensor_pm25_ug_m3,sensor_pm10_ug_m3,sensor_co_ppm,sensor_no2_ppb,sensor_so2_ppb,sensor_o3_ppb,fire_occurrence,region,sensor_co2_ppm
0,31.5,78.00,2024-04-01,3.343506,38.177402,1.085866,16.383268,676.0698,0.000000,0.354882,...,3412.5552,15.528826,21.727829,0.294739,NaN,NaN,NaN,0,31.5N_78.0E,409.360472
1,31.5,78.00,2024-04-01,3.343506,38.177402,1.085866,16.383268,676.0698,0.000000,0.354882,...,3412.5552,20.134280,26.915130,0.468569,NaN,NaN,NaN,0,31.5N_78.0E,412.073682
2,31.5,78.75,2024-04-01,-8.402588,58.738457,1.621595,34.400845,563.8798,0.008106,0.073632,...,4840.6196,9.785140,13.498948,0.741145,NaN,NaN,NaN,0,31.5N_78.8E,418.883587
3,31.5,78.75,2024-04-01,-8.402588,58.738457,1.621595,34.400845,563.8798,0.008106,0.073632,...,4840.6196,12.504870,16.262789,0.859478,NaN,NaN,NaN,0,31.5N_78.8E,429.888929
4,31.5,79.50,2024-04-01,-2.902588,50.969078,2.591617,13.781706,607.0498,0.001431,0.096490,...,4261.4930,6.009259,8.611550,0.361983,NaN,NaN,NaN,0,31.5N_79.5E,407.221005


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2240 entries, 0 to 2239
Data columns (total 22 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   latitude                 2240 non-null   float64
 1   longitude                2240 non-null   float64
 2   date                     2240 non-null   object 
 3   air_temp_c               2240 non-null   float64
 4   rel_humidity_pct         2240 non-null   float64
 5   wind_speed_kmh           2240 non-null   float64
 6   wind_gust_kmh            2240 non-null   float64
 7   surface_pressure_hpa     2240 non-null   float64
 8   precipitation_mm         2240 non-null   float64
 9   soil_moisture_vol        2240 non-null   float64
 10  solar_radiation_mj_m2    2240 non-null   float64
 11  boundary_layer_height_m  2240 non-null   float64
 12  elevation_m              2240 non-null   float64
 13  sensor_pm25_ug_m3        2240 non-null   float64
 14  sensor_pm10_ug_m3       

In [23]:
df=df[['air_temp_c','rel_humidity_pct','wind_speed_kmh','surface_pressure_hpa','soil_moisture_vol','fire_occurrence']]


In [24]:
df.head()

,air_temp_c,rel_humidity_pct,wind_speed_kmh,surface_pressure_hpa,soil_moisture_vol,fire_occurrence
0,3.343506,38.177402,1.085866,676.0698,0.354882,0
1,3.343506,38.177402,1.085866,676.0698,0.354882,0
2,-8.402588,58.738457,1.621595,563.8798,0.073632,0
3,-8.402588,58.738457,1.621595,563.8798,0.073632,0
4,-2.902588,50.969078,2.591617,607.0498,0.096490,0


In [25]:
df.describe()

,air_temp_c,rel_humidity_pct,wind_speed_kmh,surface_pressure_hpa,soil_moisture_vol,fire_occurrence
count,2240.000000,2240.000000,2240.000000,2240.000000,2240.000000,2240.000000
mean,12.487976,48.169823,6.748339,739.494773,0.238031,0.107143
std,14.821011,18.516420,4.632299,157.487676,0.090091,0.309364
min,-12.238434,13.412465,0.232148,526.495240,0.073358,0.000000
25%,-1.925552,33.305529,3.255181,578.830085,0.157527,0.000000
50%,12.880768,43.616461,5.567100,730.926380,0.252603,0.000000
75%,26.008880,60.977660,9.141808,874.335230,0.312567,0.000000
max,37.112946,97.825710,26.809956,988.069100,0.402044,1.000000


In [8]:
X=df.drop(columns=['fire_occurrence'])
y=df['fire_occurrence']

In [26]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier,AdaBoostClassifier,StackingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, recall_score, precision_score, f1_score,balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold,GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier
from sklearn.metrics import make_scorer, fbeta_score
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
numeric_features=['air_temp_c','rel_humidity_pct','wind_speed_kmh','surface_pressure_hpa','soil_moisture_vol']

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())])
model_pipeline_xgb = Pipeline(steps=[
    ('preprocessor', ColumnTransformer(transformers=[
        ('num', numeric_transformer, numeric_features)
    ])),
    ('classifier', XGBClassifier(use_label_encoder=False, eval_metric='mlogloss',learning_rate=0.01, n_estimators=50, max_depth=5))
])
#applying the model pipeline to the training data
model_pipeline_xgb.fit(X_train, y_train)
#applying the grid search to find the best hyperparameters for the model
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
param_grid = {
    'classifier__n_estimators': [50, 100, 200],
    'classifier__max_depth': [3, 5, 7],
    'classifier__learning_rate': [0.01, 0.1, 0.2]
}
f2_scorer = make_scorer(fbeta_score, beta=2, average='weighted')
grid_search = GridSearchCV(model_pipeline_xgb, param_grid, cv=5, scoring=f2_scorer, n_jobs=-1)
grid_search.fit(X_train, y_train)


c:\Users\LOQ\AppData\Local\Programs\Python\Python310\lib\site-packages\xgboost\core.py:158: UserWarning: [15:05:06] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0c55ff5f71b100e98-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
c:\Users\LOQ\AppData\Local\Programs\Python\Python310\lib\site-packages\xgboost\core.py:158: UserWarning: [15:05:20] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0c55ff5f71b100e98-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


,estimator,"Pipeline(step...=None, ...))])"
,param_grid,"{'classifier__learning_rate': [0.01, 0.1, ...], 'classifier__max_depth': [3, 5, ...], 'classifier__n_estimators': [50, 100, ...]}"
,scoring,make_scorer(f...rage=weighted)
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('num', ...)]"


In [27]:
#evaluating the model on the test set
y_pred = grid_search.predict(X_test)
print("Classification Report:\n", classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Balanced Accuracy Score:", balanced_accuracy_score(y_test, y_pred))

Classification Report:
               precision    recall  f1-score   support

           0       0.99      1.00      1.00       390
           1       0.98      0.97      0.97        58

    accuracy                           0.99       448
   macro avg       0.99      0.98      0.99       448
weighted avg       0.99      0.99      0.99       448

Confusion Matrix:
 [[389   1]
 [  2  56]]
Balanced Accuracy Score: 0.981476569407604


In [28]:
# training by AdaBoostClassifier
model_pipeline_ada = Pipeline(steps=[
    ('preprocessor', ColumnTransformer(transformers=[('num', numeric_transformer, numeric_features)])),
    ('classifier', AdaBoostClassifier(n_estimators=200, learning_rate=0.1, random_state=42,algorithm='deprecated'))
])
model_pipeline_ada.fit(X_train, y_train)
#finding the best hyperparameters for AdaBoostClassifier using grid search
param_grid_ada = {
        'classifier__n_estimators': [50, 100, 200],
        'classifier__learning_rate': [0.01, 0.1, 1.0]
}
grid_search_ada = GridSearchCV(model_pipeline_ada, param_grid_ada, cv=5, scoring=f2_scorer, n_jobs=-1)
grid_search_ada.fit(X_train, y_train)

,estimator,Pipeline(step...m_state=42))])
,param_grid,"{'classifier__learning_rate': [0.01, 0.1, ...], 'classifier__n_estimators': [50, 100, ...]}"
,scoring,make_scorer(f...rage=weighted)
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('num', ...)]"


In [29]:
#evaluating the AdaBoostClassifier model on the test set
y_pred_ada = grid_search_ada.predict(X_test)
print("Classification Report:\n", classification_report(y_test, y_pred_ada))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_ada))
print("Balanced Accuracy Score:", balanced_accuracy_score(y_test, y_pred_ada))

Classification Report:
               precision    recall  f1-score   support

           0       0.99      0.99      0.99       390
           1       0.95      0.95      0.95        58

    accuracy                           0.99       448
   macro avg       0.97      0.97      0.97       448
weighted avg       0.99      0.99      0.99       448

Confusion Matrix:
 [[387   3]
 [  3  55]]
Balanced Accuracy Score: 0.9702917771883289


In [30]:
#training with GradientBoostingClassifier
model_pipeline_gb = Pipeline(steps=[
    ('preprocessor', ColumnTransformer(transformers=[('num', numeric_transformer, numeric_features)])),
    ('classifier', GradientBoostingClassifier(n_estimators=200, learning_rate=0.1, max_depth=5, random_state=42))
])
model_pipeline_gb.fit(X_train, y_train)
param_grid_gb = {
    'classifier__n_estimators': [50, 100, 200],
    'classifier__learning_rate': [0.01, 0.1, 0.2],
    'classifier__max_depth': [3, 5, 7],
    
}
grid_search_gb = GridSearchCV(model_pipeline_gb, param_grid_gb, cv=5, scoring=f2_scorer, n_jobs=-1)
grid_search_gb.fit(X_train, y_train)

,estimator,Pipeline(step...m_state=42))])
,param_grid,"{'classifier__learning_rate': [0.01, 0.1, ...], 'classifier__max_depth': [3, 5, ...], 'classifier__n_estimators': [50, 100, ...]}"
,scoring,make_scorer(f...rage=weighted)
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('num', ...)]"


In [31]:
#evaluating the gradient boosting model on the test set
y_pred_gb = grid_search_gb.predict(X_test)
print("Classification Report:\n", classification_report(y_test, y_pred_gb))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_gb))
print("Balanced Accuracy Score:", balanced_accuracy_score(y_test, y_pred_gb))


Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00       390
           1       0.98      1.00      0.99        58

    accuracy                           1.00       448
   macro avg       0.99      1.00      1.00       448
weighted avg       1.00      1.00      1.00       448

Confusion Matrix:
 [[389   1]
 [  0  58]]
Balanced Accuracy Score: 0.9987179487179487


In [32]:
#training with random forest classifier
model_pipeline_rf = Pipeline(steps=[
    ('preprocessor', ColumnTransformer(transformers=[('num', numeric_transformer, numeric_features)])),
    ('classifier', RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42))
])
model_pipeline_rf.fit(X_train, y_train)
param_grid_rf = {
    'classifier__n_estimators': [50, 100, 200],
    'classifier__max_depth': [3, 5, 7],
    'classifier__min_samples_split': [2, 5, 10],
    'classifier__min_samples_leaf': [1, 2, 4],
}
grid_search_rf = GridSearchCV(model_pipeline_rf, param_grid_rf, cv=5, scoring=f2_scorer, n_jobs=-1)
grid_search_rf.fit(X_train, y_train)

,estimator,Pipeline(step...m_state=42))])
,param_grid,"{'classifier__max_depth': [3, 5, ...], 'classifier__min_samples_leaf': [1, 2, ...], 'classifier__min_samples_split': [2, 5, ...], 'classifier__n_estimators': [50, 100, ...]}"
,scoring,make_scorer(f...rage=weighted)
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('num', ...)]"


In [33]:
#evalauting the random forest model on the test set
y_pred_rf = grid_search_rf.predict(X_test)
print("Classification Report:\n", classification_report(y_test, y_pred_rf))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_rf))
print("Balanced Accuracy Score:", balanced_accuracy_score(y_test, y_pred_rf))

Classification Report:
               precision    recall  f1-score   support

           0       0.99      0.99      0.99       390
           1       0.93      0.97      0.95        58

    accuracy                           0.99       448
   macro avg       0.96      0.98      0.97       448
weighted avg       0.99      0.99      0.99       448

Confusion Matrix:
 [[386   4]
 [  2  56]]
Balanced Accuracy Score: 0.9776304155614501


In [34]:
#now training by StackingClassifier
base_estimators = [
    ('ada', AdaBoostClassifier(n_estimators=200, learning_rate=0.1, random_state=42,algorithm='deprecated')),
    ('gb', GradientBoostingClassifier(n_estimators=200, learning_rate=0.1, max_depth=5, random_state=42)),
    ('xgb', XGBClassifier(use_label_encoder=False, eval_metric='mlogloss',learning_rate=0.01, n_estimators=50, max_depth=5))
]
meta_estimator = RandomForestClassifier(max_depth=7, n_estimators=50, min_samples_split=5, min_samples_leaf=2,random_state=42)
stacking_model = StackingClassifier(estimators=base_estimators, final_estimator=meta_estimator, cv=5, n_jobs=-1)
stacking_model.fit(X_train, y_train)
param_grid_stacking = {
    'final_estimator__n_estimators': [50, 100, 200],
    'final_estimator__max_depth': [3, 5, 7],
}
grid_search_stacking = GridSearchCV(stacking_model, param_grid_stacking, cv=5, scoring=f2_scorer, n_jobs=-1)
grid_search_stacking.fit(X_train, y_train)

,estimator,StackingClass... n_jobs=-1)
,param_grid,"{'final_estimator__max_depth': [3, 5, ...], 'final_estimator__n_estimators': [50, 100, ...]}"
,scoring,make_scorer(f...rage=weighted)
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,estimator,None


In [35]:
#evaluating the stacking model on the test set
y_pred_stacking = grid_search_stacking.predict(X_test)
print("Classification Report:\n", classification_report(y_test, y_pred_stacking))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_stacking))
print("Balanced Accuracy Score:", balanced_accuracy_score(y_test, y_pred_stacking))


Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00       390
           1       0.98      0.98      0.98        58

    accuracy                           1.00       448
   macro avg       0.99      0.99      0.99       448
weighted avg       1.00      1.00      1.00       448

Confusion Matrix:
 [[389   1]
 [  1  57]]
Balanced Accuracy Score: 0.9900972590627763


In [37]:
import numpy as np
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import precision_recall_curve, recall_score, precision_score, fbeta_score

def get_optimal_fbeta_threshold_and_metrics(model, X_train, y_train, X_test, y_test, beta=2):
    """
    Finds the optimal threshold on training OOF probabilities by maximizing the F-beta score,
    then evaluates that custom threshold on the test set.
    """
    # 1. Get out-of-fold probabilities on the training set
    oof_probs = cross_val_predict(
        model, X_train, y_train, cv=5, method="predict_proba", n_jobs=-1
    )[:, 1]   

    # 2. Generate precision-recall curve pairs
    # we use unique thresholds to map out predictions efficiently
    precisions, recalls, thresholds = precision_recall_curve(y_train, oof_probs, pos_label=1)

    # 3. Find the best threshold using scikit-learn's fbeta_score directly
    fbeta_scores = []
    for thresh in thresholds:
        # Vectorized thresholding step
        train_preds_at_thresh = (oof_probs >= thresh).astype(int)
        # Using binary average to track your target 'fire_occurrence' class performance directly
        score = fbeta_score(y_train, train_preds_at_thresh, beta=beta, average='binary', zero_division=0)
        fbeta_scores.append(score)
        
    fbeta_scores = np.array(fbeta_scores)
    best_idx = np.argmax(fbeta_scores)
    best_threshold = thresholds[best_idx]
    train_expected_fbeta = fbeta_scores[best_idx]

    # 4. Evaluate this custom threshold on the TEST set
    test_probs = model.predict_proba(X_test)[:, 1]
    test_preds = (test_probs >= best_threshold).astype(int)
    
    test_recall = recall_score(y_test, test_preds, pos_label=1)
    test_precision = precision_score(y_test, test_preds, pos_label=1, zero_division=0)
    test_fbeta_result = fbeta_score(y_test, test_preds, beta=beta, average='binary', zero_division=0)

    return {
        "threshold": best_threshold,
        "train_expected_fbeta": train_expected_fbeta,
        "test_recall": test_recall,
        "test_precision": test_precision,
        "test_fbeta": test_fbeta_result
    }

# --- INTEGRATION LOOP ---

model_results = {
    "XGBoost": grid_search,
    "AdaBoost": grid_search_ada,
    "GradientBoosting": grid_search_gb,
    "RandomForest": grid_search_rf,
    "StackingClassifier": grid_search_stacking,
}

custom_metrics = {}
BETA_VALUE = 2  # Adjust this to change how much you prioritize recall over precision

print(f"Evaluating models using optimized F-{BETA_VALUE} thresholds:\n")

for name, fitted_search in model_results.items():
    best_est = fitted_search.best_estimator_
    
    # Run the generic F-beta threshold optimization function
    metrics = get_optimal_fbeta_threshold_and_metrics(best_est, X_train, y_train, X_test, y_test, beta=BETA_VALUE)
    custom_metrics[name] = metrics
    
    print(f"{name:20s} -> Best Thresh: {metrics['threshold']:.3f} | Train F{BETA_VALUE}: {metrics['train_expected_fbeta']:.4f} | Test Recall: {metrics['test_recall']:.4f} | Test Precision: {metrics['test_precision']:.4f}")

# Select the winner based on the highest achieved test F-beta score
best_model_name = max(custom_metrics, key=lambda k: custom_metrics[k]["test_fbeta"])
winner = custom_metrics[best_model_name]

print(f"\n>>> BEST MODEL BY OPERATIONAL F-{BETA_VALUE} SCORE: {best_model_name}")
print(f"    Operational Threshold: {winner['threshold']:.3f}")
print(f"    Final Test F-{BETA_VALUE} Score:  {winner['test_fbeta']:.4f}")
print(f"    Final Test Recall:     {winner['test_recall']:.4f}")
print(f"    Final Test Precision:  {winner['test_precision']:.4f}")

Evaluating models using optimized F-2 thresholds:

XGBoost              -> Best Thresh: 0.265 | Train F2: 0.9686 | Test Recall: 0.9655 | Test Precision: 0.9655
AdaBoost             -> Best Thresh: 0.472 | Train F2: 0.9401 | Test Recall: 1.0000 | Test Precision: 0.8169
GradientBoosting     -> Best Thresh: 0.373 | Train F2: 0.9781 | Test Recall: 1.0000 | Test Precision: 0.9831
RandomForest         -> Best Thresh: 0.508 | Train F2: 0.9697 | Test Recall: 0.9655 | Test Precision: 0.9333
StackingClassifier   -> Best Thresh: 0.538 | Train F2: 0.9728 | Test Recall: 0.9828 | Test Precision: 0.9828

>>> BEST MODEL BY OPERATIONAL F-2 SCORE: GradientBoosting
    Operational Threshold: 0.373
    Final Test F-2 Score:  0.9966
    Final Test Recall:     1.0000
    Final Test Precision:  0.9831


In [39]:
teacher_pipeline=grid_search_gb.best_estimator_
teacher_pipeline.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [40]:
soft_labels = teacher_pipeline.predict_proba(X_train)

In [41]:
import tensorflow as tf
from tensorflow.keras import layers, models
student_model = models.Sequential([
        layers.Input(shape=(X_train.shape[1],)),
        layers.Dense(64, activation='relu'),
        layers.Dense(32, activation='relu'),
        layers.Dense(soft_labels.shape[1], activation='softmax')
])
student_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
student_model.fit(X_train, soft_labels, epochs=50, batch_size=32, validation_split=0.2)

Epoch 1/50
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8465 - loss: 1.7622 - val_accuracy: 0.8719 - val_loss: 0.8607
Epoch 2/50
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8618 - loss: 0.6828 - val_accuracy: 0.8552 - val_loss: 0.2844
Epoch 3/50
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8730 - loss: 0.4346 - val_accuracy: 0.8691 - val_loss: 0.3182
Epoch 4/50
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8863 - loss: 0.2739 - val_accuracy: 0.8719 - val_loss: 0.3823
Epoch 5/50
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8883 - loss: 0.2930 - val_accuracy: 0.8719 - val_loss: 0.5406
Epoch 6/50
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8542 - loss: 1.0063 - val_accuracy: 0.8719 - val_loss: 2.9824
Epoch 7/50
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8779 - loss: 0.7633 - val_accuracy: 0.8134 - val_loss: 0.5890
Epoch 8/50
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9079 - loss: 0.2752 - val_accuracy: 0.9081 - val_loss:

In [42]:
convertor=tf.lite.TFLiteConverter.from_keras_model(student_model)
tflite_model = convertor.convert()

with open("forest_fire_predict_model_final.tflite", "wb") as f:
    f.write(tflite_model)

INFO:tensorflow:Assets written to: C:\Users\LOQ\AppData\Local\Temp\tmpdsfm0j06\assets


INFO:tensorflow:Assets written to: C:\Users\LOQ\AppData\Local\Temp\tmpdsfm0j06\assets


Saved artifact at 'C:\Users\LOQ\AppData\Local\Temp\tmpdsfm0j06'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 9), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 2), dtype=tf.float32, name=None)
Captures:
  2302475820672: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2302475825600: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2302476059920: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2302476062208: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2302476060624: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2302476064672: TensorSpec(shape=(), dtype=tf.resource, name=None)
